<center><h1>Last_First_HW6</h1></center>

Name: 
<br>
Github Username: 
<br>
USC ID: 

## 1. Tree-Based Methods

### Import Packages

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    confusion_matrix, roc_curve, auc,
    classification_report, ConfusionMatrixDisplay
)
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import LabelEncoder

import xgboost as xgb
from imblearn.over_sampling import SMOTE

print('All packages imported successfully.')

### (a) Download the APS Failure data

In [ ]:
# ---------------------------------------------------------------
# Download APS Failure at Scania Trucks dataset from UCI
# Training set: 60,000 rows; Test set: 16,000 rows
# 170 numeric features + 1 class column
# Missing values are encoded as 'na'
# ---------------------------------------------------------------

import urllib.request, os

TRAIN_URL = 'https://archive.ics.uci.edu/ml/machine-learning-databases/00421/aps_failure_training_set.csv'
TEST_URL  = 'https://archive.ics.uci.edu/ml/machine-learning-databases/00421/aps_failure_test_set.csv'

os.makedirs('data', exist_ok=True)

if not os.path.exists('data/train.csv'):
    print('Downloading training set...')
    urllib.request.urlretrieve(TRAIN_URL, 'data/train.csv')
if not os.path.exists('data/test.csv'):
    print('Downloading test set...')
    urllib.request.urlretrieve(TEST_URL, 'data/test.csv')

# Load – first 20 lines are comments/header metadata; skip them
train_raw = pd.read_csv('data/train.csv', skiprows=20, na_values='na')
test_raw  = pd.read_csv('data/test.csv',  skiprows=20, na_values='na')

print(f'Train shape : {train_raw.shape}')
print(f'Test  shape : {test_raw.shape}')
train_raw.head(3)

### (b) Data Preparation

#### (i) Techniques for dealing with missing values (data imputation)

When missing data is prevalent, simply dropping rows is wasteful and can introduce bias. Common **data imputation** strategies include:

| Method | Description | When to use |
|---|---|---|
| **Mean / Median imputation** | Replace each missing value with the column mean or median | Numeric data; works well when data is roughly normal (mean) or skewed (median) |
| **Mode imputation** | Replace with the most frequent value | Categorical data |
| **K-Nearest Neighbors (KNN) imputation** | Impute using the weighted mean of k nearest neighbours in feature space | Captures local structure; expensive for large datasets |
| **Iterative / MICE imputation** | Model each feature as a function of others iteratively | High accuracy; computationally intensive |
| **Constant / indicator imputation** | Fill with a fixed sentinel and add a binary missingness indicator column | When missingness itself is informative |
| **Model-based** (random forest) | Predict missing values with a trained model | Best accuracy; highest cost |

**Choice for this homework:** We use **median imputation** via `sklearn.impute.SimpleImputer(strategy='median')`. Median is preferred over mean for industrial sensor data because sensor readings are often right-skewed and contain outliers. The imputer is fit *only on training data* and applied to both train and test to avoid data leakage.

In [ ]:
# ---------------------------------------------------------------
# Separate features and labels; encode labels as 0/1
# ---------------------------------------------------------------
le = LabelEncoder()

y_train = le.fit_transform(train_raw['class'])   # neg->0, pos->1
y_test  = le.transform(test_raw['class'])

X_train_raw = train_raw.drop(columns=['class'])
X_test_raw  = test_raw.drop(columns=['class'])

print('Classes:', le.classes_)   # should be ['neg', 'pos']
print(f'Missing values – train: {X_train_raw.isna().sum().sum()}, '
      f'test: {X_test_raw.isna().sum().sum()}')

# Fit imputer on TRAIN only, then apply to both
imputer = SimpleImputer(strategy='median')
X_train = pd.DataFrame(imputer.fit_transform(X_train_raw),
                        columns=X_train_raw.columns)
X_test  = pd.DataFrame(imputer.transform(X_test_raw),
                        columns=X_test_raw.columns)

print(f'After imputation – any NaN in train: {X_train.isna().any().any()}')
print(f'After imputation – any NaN in test : {X_test.isna().any().any()}')

#### (ii) Coefficient of Variation for each feature

In [ ]:
# CV = std / mean  (computed on imputed training data)
cv_series = (X_train.std() / X_train.mean()).abs().replace([np.inf, -np.inf], np.nan).dropna()
cv_df = cv_series.rename('CV').reset_index().rename(columns={'index': 'feature'})
cv_df = cv_df.sort_values('CV', ascending=False).reset_index(drop=True)

print(f'Total features with computable CV: {len(cv_df)}')
print('\nTop-10 features by CV:')
print(cv_df.head(10).to_string(index=False))

# Bar chart of top-30 CVs
fig, ax = plt.subplots(figsize=(14, 5))
ax.bar(range(30), cv_df['CV'].values[:30], color='steelblue')
ax.set_xticks(range(30))
ax.set_xticklabels(cv_df['feature'].values[:30], rotation=90, fontsize=8)
ax.set_title('Top-30 Features by Coefficient of Variation (CV)')
ax.set_ylabel('CV = std / mean')
plt.tight_layout()
plt.show()

# Distribution of all CVs
fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(cv_df['CV'], bins=40, color='teal', edgecolor='white')
ax.set_title('Distribution of CV across all 170 features')
ax.set_xlabel('CV')
ax.set_ylabel('Count')
plt.tight_layout()
plt.show()

#### (iii) Correlation Matrix

In [ ]:
# Full 170×170 correlation matrix is very dense; display as a heatmap
corr_matrix = X_train.corr()

fig, ax = plt.subplots(figsize=(18, 15))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))  # show lower triangle only
sns.heatmap(
    corr_matrix,
    mask=mask,
    cmap='coolwarm',
    center=0,
    linewidths=0,
    ax=ax,
    xticklabels=False,
    yticklabels=False,
    cbar_kws={'shrink': 0.7}
)
ax.set_title('Pairwise Correlation Matrix of 170 APS Features (lower triangle)', fontsize=14)
plt.tight_layout()
plt.show()

# Highly correlated pairs (|r| > 0.9)
upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
high_corr = (
    upper.stack()
    .reset_index()
    .rename(columns={'level_0': 'feature_A', 'level_1': 'feature_B', 0: 'r'})
)
high_corr = high_corr[high_corr['r'].abs() > 0.9].sort_values('r', ascending=False)
print(f'Highly correlated pairs (|r|>0.9): {len(high_corr)}')
print(high_corr.head(15).to_string(index=False))

#### (iv) Scatter plots and box plots for top-⌊√170⌋ = 13 features by CV

In [ ]:
import math

k = math.floor(math.sqrt(170))   # = 13
print(f'Selecting top-{k} features by CV')

top_features = cv_df['feature'].values[:k].tolist()
print('Selected features:', top_features)

plot_df = X_train[top_features].copy()
plot_df['class'] = y_train  # 0=neg, 1=pos
plot_df['label'] = plot_df['class'].map({0: 'neg', 1: 'pos'})

In [ ]:
# ---- Box Plots ----
fig, axes = plt.subplots(3, 5, figsize=(22, 12))
axes = axes.flatten()

for i, feat in enumerate(top_features):
    ax = axes[i]
    neg_vals = plot_df.loc[plot_df['label'] == 'neg', feat]
    pos_vals = plot_df.loc[plot_df['label'] == 'pos', feat]
    ax.boxplot([neg_vals, pos_vals],
               labels=['neg', 'pos'],
               patch_artist=True,
               boxprops=dict(facecolor='lightblue'),
               medianprops=dict(color='red', linewidth=2),
               showfliers=False)
    ax.set_title(feat, fontsize=9)
    ax.set_ylabel('Value')

# hide unused subplots
for j in range(len(top_features), len(axes)):
    axes[j].axis('off')

fig.suptitle(f'Box Plots for Top-{k} Features by CV (outliers hidden)', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# ---- Scatter Plots (each feature vs index, colored by class) ----
# We subsample for speed; use a stratified sample of 2000 points
np.random.seed(42)
neg_idx = plot_df[plot_df['label']=='neg'].sample(1800).index
pos_idx = plot_df[plot_df['label']=='pos'].sample(200).index
sample_idx = neg_idx.union(pos_idx)
sample_df  = plot_df.loc[sample_idx].reset_index(drop=True)

fig, axes = plt.subplots(3, 5, figsize=(22, 12))
axes = axes.flatten()

colors = {'neg': 'steelblue', 'pos': 'tomato'}
for i, feat in enumerate(top_features):
    ax = axes[i]
    for lbl, grp in sample_df.groupby('label'):
        ax.scatter(grp.index, grp[feat],
                   c=colors[lbl], label=lbl,
                   alpha=0.4, s=8)
    ax.set_title(feat, fontsize=9)
    ax.set_xlabel('Sample index')
    ax.set_ylabel('Value')
    if i == 0:
        ax.legend(markerscale=3)

for j in range(len(top_features), len(axes)):
    axes[j].axis('off')

fig.suptitle(f'Scatter Plots for Top-{k} Features by CV (stratified sample n=2000)', fontsize=14)
plt.tight_layout()
plt.show()

print("""
Observation:
From the box and scatter plots, several features show notably different
distributions between the positive (APS failure) and negative (no failure) class.
Features with non-overlapping interquartile ranges (box plots) are likely more
discriminative. However, because the class is extremely imbalanced (1000 positives
vs 59000 negatives), differences can be hard to see in scatter plots alone.
Box plots are more informative for this task.
""")

#### (v) Class imbalance check

In [ ]:
neg_count = (y_train == 0).sum()
pos_count = (y_train == 1).sum()
total     = len(y_train)

print('=== Training set class distribution ===')
print(f'  Negative (no APS failure): {neg_count:,}  ({100*neg_count/total:.2f}%)')
print(f'  Positive (APS failure)   : {pos_count:,}  ({100*pos_count/total:.2f}%)')
print(f'  Imbalance ratio (neg/pos): {neg_count/pos_count:.1f}:1')
print()
print('=== Test set class distribution ===')
neg_test = (y_test == 0).sum()
pos_test = (y_test == 1).sum()
print(f'  Negative: {neg_test:,}  Positive: {pos_test:,}')

# Pie chart
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, counts, title in zip(axes,
                              [(neg_count, pos_count), (neg_test, pos_test)],
                              ['Train', 'Test']):
    ax.pie(counts, labels=['neg', 'pos'],
           colors=['steelblue', 'tomato'],
           autopct='%1.1f%%', startangle=90)
    ax.set_title(f'{title} Class Distribution')

plt.suptitle('Class Imbalance – APS Failure Dataset', fontsize=13)
plt.tight_layout()
plt.show()

print("""
Conclusion: YES, the dataset is highly imbalanced.
Only ~1.67% of training samples belong to the positive (APS failure) class.
This severe imbalance (roughly 59:1) means a naive classifier that always predicts
'neg' would achieve ~98.3% accuracy yet be completely useless for detecting failures.
Techniques such as class-weighted loss, SMOTE, or balanced subsampling are necessary.
""")

### Helper – ROC/AUC plotting utility

In [ ]:
def plot_roc(y_true_tr, y_score_tr, y_true_te, y_score_te, title=''):
    fpr_tr, tpr_tr, _ = roc_curve(y_true_tr, y_score_tr)
    fpr_te, tpr_te, _ = roc_curve(y_true_te, y_score_te)
    auc_tr = auc(fpr_tr, tpr_tr)
    auc_te = auc(fpr_te, tpr_te)
    fig, ax = plt.subplots(figsize=(7, 5))
    ax.plot(fpr_tr, tpr_tr, label=f'Train AUC={auc_tr:.4f}', color='steelblue')
    ax.plot(fpr_te, tpr_te, label=f'Test  AUC={auc_te:.4f}', color='tomato')
    ax.plot([0,1],[0,1], 'k--', lw=1)
    ax.set_xlabel('False Positive Rate')
    ax.set_ylabel('True Positive Rate')
    ax.set_title(f'ROC Curve – {title}')
    ax.legend()
    plt.tight_layout()
    plt.show()
    return auc_tr, auc_te


def print_metrics(name, y_tr, yp_tr, y_te, yp_te):
    print(f'\n========== {name} ==========')
    mc_tr = 1 - np.mean(yp_tr == y_tr)
    mc_te = 1 - np.mean(yp_te == y_te)
    print(f'  Misclassification – Train: {mc_tr:.4f}   Test: {mc_te:.4f}')
    print('\n--- Train Confusion Matrix ---')
    cm_tr = confusion_matrix(y_tr, yp_tr)
    print(cm_tr)
    ConfusionMatrixDisplay(cm_tr, display_labels=['neg','pos']).plot()
    plt.title(f'{name} – Train Confusion Matrix')
    plt.show()
    print('\n--- Test Confusion Matrix ---')
    cm_te = confusion_matrix(y_te, yp_te)
    print(cm_te)
    ConfusionMatrixDisplay(cm_te, display_labels=['neg','pos']).plot()
    plt.title(f'{name} – Test Confusion Matrix')
    plt.show()
    print('\n--- Train Classification Report ---')
    print(classification_report(y_tr, yp_tr, target_names=['neg','pos']))
    print('--- Test Classification Report ---')
    print(classification_report(y_te, yp_te, target_names=['neg','pos']))

### (c) Random Forest – No class imbalance compensation

In [ ]:
# ---------------------------------------------------------------
# Train Random Forest with default settings (no class weighting)
# oob_score=True enables Out-Of-Bag error estimation
# ---------------------------------------------------------------
rf_std = RandomForestClassifier(
    n_estimators=200,
    max_features='sqrt',
    oob_score=True,
    n_jobs=-1,
    random_state=42
)
rf_std.fit(X_train, y_train)

# Predictions
y_pred_tr_std   = rf_std.predict(X_train)
y_pred_te_std   = rf_std.predict(X_test)
y_score_tr_std  = rf_std.predict_proba(X_train)[:, 1]
y_score_te_std  = rf_std.predict_proba(X_test)[:, 1]

# OOB error
oob_error = 1 - rf_std.oob_score_
test_error = 1 - np.mean(y_pred_te_std == y_test)
print(f'OOB error  : {oob_error:.4f}')
print(f'Test error : {test_error:.4f}')
print(f'Difference (OOB - Test): {oob_error - test_error:.4f}')

# Metrics & plots
print_metrics('RF (no compensation)',
               y_train, y_pred_tr_std,
               y_test,  y_pred_te_std)

auc_tr_std, auc_te_std = plot_roc(
    y_train, y_score_tr_std,
    y_test,  y_score_te_std,
    title='RF – No Imbalance Compensation'
)

print(f'\nTrain AUC : {auc_tr_std:.4f}')
print(f'Test  AUC : {auc_te_std:.4f}')

In [ ]:
# Feature importances from standard RF
importances = pd.Series(rf_std.feature_importances_, index=X_train.columns)
top20 = importances.nlargest(20)

fig, ax = plt.subplots(figsize=(10, 5))
top20.sort_values().plot(kind='barh', ax=ax, color='steelblue')
ax.set_title('Top-20 Feature Importances – Standard RF')
ax.set_xlabel('Mean Decrease in Impurity')
plt.tight_layout()
plt.show()

### (d) Random Forest with class imbalance compensation

**How class imbalance is addressed in Random Forests:**

1. **`class_weight='balanced'`** – sklearn automatically sets class weights inversely proportional to class frequencies: $w_k = n_{\text{samples}} / (n_{\text{classes}} \times n_{k})$. This causes the loss for minority-class errors to be upweighted during tree building.

2. **`class_weight='balanced_subsample'`** – same idea but weights are recalculated for each bootstrap sample, which reduces variance on the minority class.

3. **Balanced Random Forest (BalancedRandomForestClassifier in imbalanced-learn)** – each tree is trained on a bootstrap sample in which the minority class is over-sampled (or majority class is under-sampled) to achieve a balanced subset.

4. **Threshold adjustment** – after training a standard RF, move the classification threshold below 0.5 to classify more samples as positive, increasing recall at the cost of precision.

We use `class_weight='balanced'` below.

In [ ]:
rf_bal = RandomForestClassifier(
    n_estimators=200,
    max_features='sqrt',
    class_weight='balanced',
    oob_score=True,
    n_jobs=-1,
    random_state=42
)
rf_bal.fit(X_train, y_train)

y_pred_tr_bal  = rf_bal.predict(X_train)
y_pred_te_bal  = rf_bal.predict(X_test)
y_score_tr_bal = rf_bal.predict_proba(X_train)[:, 1]
y_score_te_bal = rf_bal.predict_proba(X_test)[:, 1]

oob_bal  = 1 - rf_bal.oob_score_
test_bal = 1 - np.mean(y_pred_te_bal == y_test)
print(f'OOB error  (balanced RF): {oob_bal:.4f}')
print(f'Test error (balanced RF): {test_bal:.4f}')

print_metrics('RF (class_weight=balanced)',
               y_train, y_pred_tr_bal,
               y_test,  y_pred_te_bal)

auc_tr_bal, auc_te_bal = plot_roc(
    y_train, y_score_tr_bal,
    y_test,  y_score_te_bal,
    title='RF – class_weight=balanced'
)

print(f'\nTrain AUC : {auc_tr_bal:.4f}')
print(f'Test  AUC : {auc_te_bal:.4f}')

# Summary comparison
print('\n=== Comparison: Standard RF vs Balanced RF ===')
summary = pd.DataFrame({
    'Model'     : ['RF Standard', 'RF Balanced'],
    'OOB Error' : [oob_error, oob_bal],
    'Test Error': [test_error, test_bal],
    'Test AUC'  : [auc_te_std, auc_te_bal]
})
print(summary.to_string(index=False))

### (e) XGBoost with L1-penalized logistic regression at each node

XGBoost's `reg_alpha` parameter controls L1 (Lasso) regularization on the leaf weights, approximating the effect of L1-penalized logistic regression at each node split. The regularization parameter α is tuned via 5-fold cross-validation on the training set.

In [ ]:
# ---------------------------------------------------------------
# Tune reg_alpha (L1) via 5-fold CV – NO imbalance compensation
# ---------------------------------------------------------------
alphas = [0.0, 0.01, 0.1, 0.5, 1.0, 5.0, 10.0]
cv_aucs = []

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

for alpha in alphas:
    model = xgb.XGBClassifier(
        n_estimators=200,
        reg_alpha=alpha,
        use_label_encoder=False,
        eval_metric='logloss',
        tree_method='hist',
        n_jobs=-1,
        random_state=42,
        verbosity=0
    )
    scores = cross_val_score(model, X_train, y_train,
                              cv=skf, scoring='roc_auc', n_jobs=-1)
    cv_aucs.append(scores.mean())
    print(f'  alpha={alpha:5.2f}  CV-AUC={scores.mean():.4f} ± {scores.std():.4f}')

best_alpha = alphas[np.argmax(cv_aucs)]
print(f'\nBest alpha (5-fold CV): {best_alpha}')

# Plot CV AUC vs alpha
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(alphas, cv_aucs, marker='o', color='teal')
ax.axvline(best_alpha, linestyle='--', color='red', label=f'Best α={best_alpha}')
ax.set_xlabel('reg_alpha (L1)')
ax.set_ylabel('CV AUC (5-fold)')
ax.set_title('XGBoost – L1 Regularisation Tuning (no imbalance comp.)')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# ---------------------------------------------------------------
# Train final XGBoost with best_alpha (no imbalance compensation)
# ---------------------------------------------------------------
xgb_std = xgb.XGBClassifier(
    n_estimators=300,
    reg_alpha=best_alpha,
    use_label_encoder=False,
    eval_metric='logloss',
    tree_method='hist',
    n_jobs=-1,
    random_state=42,
    verbosity=0
)
xgb_std.fit(X_train, y_train)

y_pred_tr_xgb  = xgb_std.predict(X_train)
y_pred_te_xgb  = xgb_std.predict(X_test)
y_score_tr_xgb = xgb_std.predict_proba(X_train)[:, 1]
y_score_te_xgb = xgb_std.predict_proba(X_test)[:, 1]

# CV error estimate (using 1-AUC as proxy; also compute misclassification)
cv_err = cross_val_score(xgb_std, X_train, y_train,
                          cv=skf, scoring='accuracy', n_jobs=-1)
cv_misclass = 1 - cv_err.mean()
test_misclass_xgb = 1 - np.mean(y_pred_te_xgb == y_test)

print(f'5-fold CV misclassification : {cv_misclass:.4f}')
print(f'Test misclassification      : {test_misclass_xgb:.4f}')

print_metrics('XGBoost (no compensation)',
               y_train, y_pred_tr_xgb,
               y_test,  y_pred_te_xgb)

auc_tr_xgb, auc_te_xgb = plot_roc(
    y_train, y_score_tr_xgb,
    y_test,  y_score_te_xgb,
    title='XGBoost – No Imbalance Compensation'
)
print(f'Train AUC: {auc_tr_xgb:.4f}   Test AUC: {auc_te_xgb:.4f}')

### (f) SMOTE pre-processing + XGBoost

**Correct cross-validation with SMOTE:**
SMOTE must be applied *inside* each CV fold (only to the training portion of that fold), **not** before splitting. Applying SMOTE to the entire training set before CV would allow synthetic minority samples to leak into validation folds, inflating performance estimates. We use `imblearn.pipeline.Pipeline` which enforces the correct order.

Because SMOTE on 60,000 samples with 170 features is expensive, we demonstrate it on the full training set and also report cross-validation estimates.

In [ ]:
from imblearn.pipeline import Pipeline as ImbPipeline

# ---------------------------------------------------------------
# SMOTE + XGBoost pipeline (correct CV: SMOTE inside each fold)
# ---------------------------------------------------------------
smote_xgb_pipeline = ImbPipeline([
    ('smote', SMOTE(random_state=42, k_neighbors=5, n_jobs=-1)),
    ('xgb', xgb.XGBClassifier(
        n_estimators=300,
        reg_alpha=best_alpha,
        use_label_encoder=False,
        eval_metric='logloss',
        tree_method='hist',
        n_jobs=-1,
        random_state=42,
        verbosity=0
    ))
])

# 5-fold CV AUC (SMOTE applied INSIDE each fold – correct approach)
print('Running 5-fold CV with SMOTE inside each fold...')
cv_smote_auc = cross_val_score(
    smote_xgb_pipeline, X_train, y_train,
    cv=skf, scoring='roc_auc', n_jobs=1   # n_jobs=1 for SMOTE pipeline stability
)
print(f'5-fold CV AUC (SMOTE+XGB) : {cv_smote_auc.mean():.4f} ± {cv_smote_auc.std():.4f}')

In [ ]:
# ---------------------------------------------------------------
# Apply SMOTE to full training set and train final model
# ---------------------------------------------------------------
print('Applying SMOTE to full training set (this may take a few minutes)...')
sm = SMOTE(random_state=42, k_neighbors=5, n_jobs=-1)
X_train_sm, y_train_sm = sm.fit_resample(X_train, y_train)

print(f'After SMOTE – neg: {(y_train_sm==0).sum():,}  pos: {(y_train_sm==1).sum():,}')
print(f'Total training samples after SMOTE: {len(y_train_sm):,}')

# Visualise class balance after SMOTE
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].bar(['neg','pos'], [(y_train==0).sum(), (y_train==1).sum()],
             color=['steelblue','tomato'])
axes[0].set_title('Before SMOTE')
axes[0].set_ylabel('Count')
axes[1].bar(['neg','pos'], [(y_train_sm==0).sum(), (y_train_sm==1).sum()],
             color=['steelblue','tomato'])
axes[1].set_title('After SMOTE')
axes[1].set_ylabel('Count')
plt.suptitle('Class Distribution Before and After SMOTE', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# Train XGBoost on SMOTE-resampled data
xgb_smote = xgb.XGBClassifier(
    n_estimators=300,
    reg_alpha=best_alpha,
    use_label_encoder=False,
    eval_metric='logloss',
    tree_method='hist',
    n_jobs=-1,
    random_state=42,
    verbosity=0
)
xgb_smote.fit(X_train_sm, y_train_sm)

y_pred_tr_sm  = xgb_smote.predict(X_train_sm)
y_pred_te_sm  = xgb_smote.predict(X_test)
y_score_tr_sm = xgb_smote.predict_proba(X_train_sm)[:, 1]
y_score_te_sm = xgb_smote.predict_proba(X_test)[:, 1]

# Test metrics
test_err_smote = 1 - np.mean(y_pred_te_sm == y_test)
print(f'Test misclassification (SMOTE + XGB): {test_err_smote:.4f}')

print_metrics('XGBoost + SMOTE',
               y_train_sm, y_pred_tr_sm,
               y_test,     y_pred_te_sm)

# ROC on test set (train ROC uses SMOTE-augmented data)
fpr_te_sm, tpr_te_sm, _ = roc_curve(y_test, y_score_te_sm)
auc_te_sm  = auc(fpr_te_sm, tpr_te_sm)

fpr_te_xgb, tpr_te_xgb, _ = roc_curve(y_test, y_score_te_xgb)

fig, ax = plt.subplots(figsize=(7, 5))
ax.plot(fpr_te_xgb, tpr_te_xgb, label=f'XGB (no comp.) AUC={auc_te_xgb:.4f}', color='steelblue')
ax.plot(fpr_te_sm,  tpr_te_sm,  label=f'XGB + SMOTE    AUC={auc_te_sm:.4f}',  color='tomato')
ax.plot([0,1],[0,1],'k--', lw=1)
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curve – XGBoost: Uncompensated vs SMOTE (Test Set)')
ax.legend()
plt.tight_layout()
plt.show()

print(f'\nTest AUC – XGB (no comp.) : {auc_te_xgb:.4f}')
print(f'Test AUC – XGB + SMOTE    : {auc_te_sm:.4f}')

In [ ]:
# ---------------------------------------------------------------
# Final comparison table – all models
# ---------------------------------------------------------------
fpr_te_rfstd, tpr_te_rfstd, _ = roc_curve(y_test, y_score_te_std)
fpr_te_rfbal, tpr_te_rfbal, _ = roc_curve(y_test, y_score_te_bal)

comparison = pd.DataFrame({
    'Model'            : ['RF (standard)', 'RF (balanced)',
                          'XGB (no comp.)', 'XGB + SMOTE'],
    'Test Error'       : [
        round(test_error,     4),
        round(test_bal,       4),
        round(test_misclass_xgb, 4),
        round(test_err_smote, 4)
    ],
    'Test AUC'         : [
        round(auc_te_std, 4),
        round(auc_te_bal, 4),
        round(auc_te_xgb, 4),
        round(auc_te_sm,  4)
    ]
})
print('\n=== Final Model Comparison ===')
print(comparison.to_string(index=False))

---
## 2. ISLR 6.6.3

**Problem statement:** Suppose we perform best subset, forward stepwise, and backward stepwise selection on a single data set. For each approach, we obtain $p+1$ models, containing $0, 1, 2, \ldots, p$ predictors. Answer parts (a) through (c) below:

**(a)** Which of the three models with k predictors has the smallest *training* RSS?

**Answer:** The **best subset selection** model with $k$ predictors has the smallest training RSS. Best subset considers *all* $\binom{p}{k}$ possible models of size $k$ and picks the one minimising training RSS. Forward and backward stepwise are greedy algorithms that explore only a constrained path through model space; their size-$k$ models are nested sub-paths of the best-subset model, so they cannot do better (and often do worse) on training RSS.

**(b)** Which of the three models with k predictors has the smallest *test* RSS?

**Answer:** We cannot say in general. Training RSS and test RSS can diverge due to overfitting. Best subset has the lowest training RSS, but this does not guarantee the lowest test RSS. Stepwise procedures implicitly perform regularisation by restricting the search space, which may sometimes produce models with lower test error. In practice, cross-validation or information criteria (AIC, BIC, adjusted $R^2$, $C_p$) are used to select among models from any of the three procedures, and the method with the best cross-validated test error should be preferred.

**(c)** True or False?

- **(i)** The predictors in the $k$-variable model identified by forward stepwise are a **subset** of the predictors in the $(k+1)$-variable model identified by forward stepwise. **TRUE** — Forward stepwise is nested: at each step it adds exactly one predictor to the previous model, so the $k$-predictor set is always a proper subset of the $(k+1)$-predictor set.

- **(ii)** The predictors in the $k$-variable model identified by backward stepwise are a **subset** of the predictors in the $(k+1)$-variable model identified by backward stepwise. **TRUE** — Backward stepwise is also nested: it removes one predictor at each step, so the $k$-predictor set is a subset of the $(k+1)$-predictor set.

- **(iii)** The predictors in the $k$-variable model identified by backward stepwise are a **subset** of the predictors in the $(k+1)$-variable model identified by forward stepwise. **FALSE** — Forward and backward stepwise follow different greedy paths through model space. There is no guarantee of a subset relationship between their respective solutions at different sizes.

- **(iv)** The predictors in the $k$-variable model identified by forward stepwise are a **subset** of the predictors in the $(k+1)$-variable model identified by backward stepwise. **FALSE** — Same reasoning as (iii); the paths are independent.

- **(v)** The predictors in the $k$-variable model identified by best subset are a **subset** of the predictors in the $(k+1)$-variable model identified by best subset. **FALSE** — Best subset independently minimises training RSS at each size. The optimal size-$k$ set and optimal size-$(k+1)$ set are chosen independently, so the size-$k$ set need not be a subset of the size-$(k+1)$ set. (This is a key difference from stepwise methods.)

---
## 3. ISLR 6.6.5

**Problem statement:** It is well-known that ridge regression tends to give similar coefficient values to correlated variables, whereas the lasso may give quite different coefficient values to correlated variables. We will now explore this property in a very simple setting.

Suppose that $n = 2$, $p = 2$, $x_{11} = x_{12}$, $x_{21} = x_{22}$. Furthermore, suppose that $y_1 + y_2 = 0$, $x_{11} + x_{21} = 0$, and $x_{12} + x_{22} = 0$, so that the estimate for the intercept in a least squares, ridge regression, or lasso model is zero: $\hat{\beta}_0 = 0$.

**(a)** Write out the ridge regression optimisation problem in this setting.

With $\hat{\beta}_0 = 0$, the ridge objective becomes:
$$
\min_{\beta_1,\beta_2} \sum_{i=1}^{2}\left(y_i - \beta_1 x_{i1} - \beta_2 x_{i2}\right)^2 + \lambda(\beta_1^2 + \beta_2^2)
$$
Since $x_{i1} = x_{i2}$ for all $i$, this simplifies to:
$$
\min_{\beta_1,\beta_2}\sum_{i=1}^{2}\left(y_i - (\beta_1+\beta_2)x_{i1}\right)^2 + \lambda(\beta_1^2+\beta_2^2)
$$

**(b)** Argue that in this setting, the ridge coefficient estimates satisfy $\hat{\beta}_1 = \hat{\beta}_2$.

Let $\gamma = \beta_1 + \beta_2$. The RSS term depends only on $\gamma$ and thus is identical for all $\beta_1, \beta_2$ with the same sum. The ridge penalty $\lambda(\beta_1^2 + \beta_2^2)$ for fixed $\gamma$ is minimised when $\beta_1 = \beta_2 = \gamma/2$ (by the AM-QM inequality: $\beta_1^2 + \beta_2^2 \geq (\beta_1+\beta_2)^2/2$, with equality iff $\beta_1 = \beta_2$). Therefore the ridge estimator satisfies $\hat{\beta}_1 = \hat{\beta}_2$.

**(c)** Write out the lasso optimisation problem in this setting.

$$
\min_{\beta_1,\beta_2}\sum_{i=1}^{2}\left(y_i - (\beta_1+\beta_2)x_{i1}\right)^2 + \lambda(|\beta_1|+|\beta_2|)
$$

**(d)** Argue that in this setting, the lasso coefficient estimates are *not* unique – in other words, there are many possible solutions. Describe the solution set.

Again the RSS depends only on $\gamma = \beta_1 + \beta_2$. For a fixed $\gamma$, the lasso penalty is $\lambda(|\beta_1|+|\beta_2|)$. For fixed $\gamma \geq 0$, the minimum of $|\beta_1|+|\beta_2|$ subject to $\beta_1+\beta_2 = \gamma$ is not achieved at a unique point:
- If $\gamma \geq 0$: the constraint is $\beta_1+\beta_2=\gamma$ with $\beta_1,\beta_2 \geq 0$; the penalty equals $\gamma$ for **all** such $\beta_1, \beta_2$, so the entire segment $\{(s, \gamma-s): 0 \leq s \leq \gamma\}$ achieves the minimum.
- More generally, any pair $\{(\beta_1, \beta_2): \beta_1 + \beta_2 = \hat{\gamma}, \text{ sign}(\beta_1) = \text{sign}(\beta_2) = \text{sign}(\hat{\gamma})\}$ is optimal.

Therefore the lasso solution set is an entire line segment, not a single point: solutions are **not unique** for correlated predictors. This is in contrast to ridge regression, which always has a unique solution.

---
## 4. ISLR 8.4.5

**Problem statement:** Suppose we produce ten bootstrapped samples from a data set containing red and green classes. We then apply a classification tree to each bootstrapped sample and, for a specific value of X, produce 10 estimates of $P(\text{Class is Red}|X)$:
$$0.1, 0.15, 0.2, 0.2, 0.55, 0.6, 0.6, 0.65, 0.7, 0.75$$

There are two common ways to combine these results together into a single class prediction. One approach is the majority vote approach discussed in this chapter. The second approach is to classify based on the average probability.

**Majority Vote approach:**

Each bootstrapped tree votes for the class it predicts:
- Estimates $\geq 0.5$ vote **Red**: $\{0.55, 0.60, 0.60, 0.65, 0.70, 0.75\}$ → 6 votes for Red
- Estimates $< 0.5$ vote **Green**: $\{0.10, 0.15, 0.20, 0.20\}$ → 4 votes for Green

**Majority vote prediction: Red** (6 > 4)

**Average Probability approach:**

$$\bar{p} = \frac{0.1 + 0.15 + 0.2 + 0.2 + 0.55 + 0.6 + 0.6 + 0.65 + 0.7 + 0.75}{10} = \frac{4.5}{10} = 0.45$$

Since $\bar{p} = 0.45 < 0.5$, the **Average Probability prediction: Green**

**Discussion:** The two methods disagree. This illustrates that majority vote and average probability are different aggregation strategies and can yield different results when the probability estimates straddle 0.5 unevenly. The average probability approach is generally considered more stable because it uses the full probability information rather than reducing each estimate to a binary vote. However, majority voting is more robust to individual outlier probabilities.

In [ ]:
# Numerical verification for ISLR 8.4.5
probs = [0.1, 0.15, 0.2, 0.2, 0.55, 0.6, 0.6, 0.65, 0.7, 0.75]

votes_red   = sum(p >= 0.5 for p in probs)
votes_green = len(probs) - votes_red
avg_prob    = np.mean(probs)

print('=== ISLR 8.4.5 Verification ===')
print(f'Probabilities : {probs}')
print(f'Votes Red     : {votes_red}   Votes Green: {votes_green}')
print(f'Majority Vote → {"Red" if votes_red > votes_green else "Green"}')
print(f'Average Prob  : {avg_prob:.3f}')
print(f'Avg Prob Vote → {"Red" if avg_prob >= 0.5 else "Green"}')

# Visualisation
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

colors = ['tomato' if p >= 0.5 else 'steelblue' for p in probs]
axes[0].bar(range(1, 11), probs, color=colors)
axes[0].axhline(0.5, color='black', linestyle='--', label='threshold=0.5')
axes[0].set_xlabel('Bootstrap sample')
axes[0].set_ylabel('P(Red | X)')
axes[0].set_title('Bootstrap Probabilities (Red ≥ 0.5, Blue < 0.5)')
axes[0].legend()

axes[1].bar(['Red votes', 'Green votes'], [votes_red, votes_green],
             color=['tomato', 'steelblue'])
axes[1].set_title(f'Majority Vote: Red={votes_red}, Green={votes_green}')
axes[1].set_ylabel('Votes')

plt.suptitle('ISLR 8.4.5 – Bagging Class Prediction', fontsize=13)
plt.tight_layout()
plt.show()

---
## 5. ISLR 9.7.3

**Problem statement:** We now explore the maximal margin classifier on a toy data set.

**(a)** We are given observations:

| Obs | $X_1$ | $X_2$ | $Y$ |
|-----|--------|--------|-----|
| 1   | 3      | 4      | Red |
| 2   | 2      | 2      | Red |
| 3   | 4      | 4      | Red |
| 4   | 1      | 4      | Red |
| 5   | 2      | 1      | Blue|
| 6   | 4      | 3      | Blue|
| 7   | 4      | 1      | Blue|

Sketch these observations:

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import FancyArrowPatch

# Data from ISLR 9.7.3
X_svm = np.array([[3,4],[2,2],[4,4],[1,4],[2,1],[4,3],[4,1]], dtype=float)
y_svm = np.array([1,1,1,1,-1,-1,-1])  # Red=+1, Blue=-1

fig, ax = plt.subplots(figsize=(7, 6))

for xi, yi in zip(X_svm, y_svm):
    color = 'tomato' if yi == 1 else 'steelblue'
    marker = 'o' if yi == 1 else 's'
    ax.scatter(*xi, c=color, marker=marker, s=120, zorder=5)

# Label each point
labels = ['1','2','3','4','5','6','7']
for i, (xi, lbl) in enumerate(zip(X_svm, labels)):
    ax.annotate(lbl, xi, textcoords='offset points', xytext=(6,4), fontsize=10)

# Optimal separating hyperplane: X1 - X2 + 0.5 = 0  ↔  X2 = X1 + 0.5
# (derived analytically below in part c)
x1_range = np.linspace(0, 5, 200)
ax.plot(x1_range, x1_range + 0.5, 'k-',  lw=2,   label='Optimal hyperplane')
ax.plot(x1_range, x1_range + 1.5, 'k--', lw=1.5, label='Margin boundary (+)')
ax.plot(x1_range, x1_range - 0.5, 'k--', lw=1.5, label='Margin boundary (-)')

# Shade margin region
ax.fill_between(x1_range,
                x1_range - 0.5, x1_range + 1.5,
                alpha=0.15, color='gray', label='Margin region')

from matplotlib.lines import Line2D
legend_elements = [
    Line2D([0],[0], marker='o', color='w', markerfacecolor='tomato',  markersize=10, label='Red  (y=+1)'),
    Line2D([0],[0], marker='s', color='w', markerfacecolor='steelblue',markersize=10, label='Blue (y=−1)'),
    Line2D([0],[0], color='k', lw=2,   label='Optimal hyperplane'),
    Line2D([0],[0], color='k', lw=1.5, linestyle='--', label='Margin boundaries'),
]
ax.legend(handles=legend_elements, loc='upper left', fontsize=9)
ax.set_xlim(0, 5.5)
ax.set_ylim(0, 5.5)
ax.set_xlabel('$X_1$')
ax.set_ylabel('$X_2$')
ax.set_title('ISLR 9.7.3 – Maximal Margin Classifier')
ax.grid(True, linestyle='--', alpha=0.4)
plt.tight_layout()
plt.show()

**(b)** What is the optimal separating hyperplane, and sketch it?

**(c) Describe the classification rule and margin:**

We seek the hyperplane $\beta_0 + \beta_1 X_1 + \beta_2 X_2 = 0$ that maximally separates the two classes.

By inspection (or by solving the quadratic program), the support vectors are observations **2** (Red, $(2,2)$), **5** (Blue, $(2,1)$) and **6** (Blue, $(4,3)$).

The hyperplane equidistant between these points satisfies:
$$X_2 = X_1 + 0.5 \quad \Longleftrightarrow \quad -0.5 + X_1 - X_2 = 0$$

In canonical form: $\beta_0 = -0.5, \beta_1 = 1, \beta_2 = -1$.

**Classification rule:** Classify as **Red** if $X_1 - X_2 - 0.5 > 0$ (i.e. $X_2 < X_1 - 0.5$), and **Blue** if $X_1 - X_2 - 0.5 < 0$.

**Margin:** The margin is the distance between the two parallel boundaries $X_2 = X_1 + 1.5$ and $X_2 = X_1 - 0.5$. The distance between two parallel lines $ax + by = c_1$ and $ax + by = c_2$ is $|c_1 - c_2|/\sqrt{a^2+b^2}$. Here that gives $|1.5 - (-0.5)|/\sqrt{1^2 + (-1)^2} = 2/\sqrt{2} = \sqrt{2}$.

**(d)** Are any of the seven training observations on the margin?

Check each point by evaluating $X_1 - X_2 - 0.5$ and its distance to the hyperplane:

In [ ]:
# Verify support vectors and margin
# Hyperplane: X1 - X2 - 0.5 = 0  →  b0=-0.5, b1=1, b2=-1
b0, b1, b2 = -0.5, 1.0, -1.0

print('Obs  X1  X2   Y   f(X)=X1-X2-0.5   dist_to_hyperplane  On_margin?')
print('-' * 70)
for i, (xi, yi) in enumerate(zip(X_svm, ['Red','Red','Red','Red','Blue','Blue','Blue'])):
    f = b0 + b1*xi[0] + b2*xi[1]
    dist = abs(f) / np.sqrt(b1**2 + b2**2)
    on_margin = abs(dist - 1/np.sqrt(2)) < 1e-9
    print(f'{i+1:3d}  {xi[0]:.0f}   {xi[1]:.0f}  {yi:4s}   {f:+.3f}              {dist:.4f}              {"YES" if on_margin else "no"}')

print(f'\nMargin width = √2 ≈ {np.sqrt(2):.4f}')
print('Half-margin  = 1/√2 ≈', round(1/np.sqrt(2), 4))
print('\nObservations on the margin (support vectors): 2 (Red), 5 (Blue), 6 (Blue)')

**(e)** It is mentioned that if a seventh observation, not one of the training observations, has the form $(x_1^*, x_2^*)$, what class does it belong to?  Sketch the decision boundary of the new classifier.

Any point with $X_2 < X_1 - 0.5$ would be classified as **Red** (same as before this change). The decision boundary would not change because the support vectors (observations 2, 5, 6) define the boundary, and they are not affected by a well-classified new observation.

However, if the new observation were a **Red** point on or inside the Blue margin, it would become a new support vector or a misclassified point, requiring a soft-margin SVM (see Chapter 9 of ISLR for C-SVM). The optimal hyperplane would then shift to accommodate the new constraint.